Imports and Libraries needed

In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

C:\Users\DELL\AppData\Local\Temp\ipykernel_9464\3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Setting AI's API

In [5]:
model= "gpt-4.1-nano"
db_name='vector_db'
load_dotenv(override=True)
openai_api_key=os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print("The api is working")
else:
    print("Key not set")

The api is working


Setting Knowledge Base / Vector DB

In [ ]:
knowledge_base="knowledge_base/**/*.md"
files =glob.glob(knowledge_base, recursive=True)
print(f"{len(files)} files found in the Knowledge base")

all_knowledge_base = ""

for fp in files:
    with open(fp, 'r',encoding='utf-8') as f:
        all_knowledge_base += f.read()
        all_knowledge_base += "\n\n"

print(f"all {len(all_knowledge_base):,} characters are found in the knowledge base")

5 files found in the Knowledge base
all 4,951 characters are f


taking all in the knowledge base then tokenize it, Count total tokens 

In [ ]:
encoding =tiktoken.encoding_for_model(model) #tokeniser corresponding to a specific model in the OpenAI API
tokens=encoding.encode(all_knowledge_base) #translating the text into numbers
token_count=len(tokens)
print(f"There are {token_count} tokens")

There are 1017 tokens


Loading all knowledge_base using langchain

In [ ]:
folders=glob.glob("knowledge_base/*")

documents=[]
for f in folders:
    doc_type=os.path.basename(f)
    loader=DirectoryLoader(f, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs= loader.load()
    for doc in folder_docs:
        doc.metadata['doc_type'] = doc_type
        documents.append(doc) #getting all content of knowledge base and putting it all in one array
print(f'loaded {len(documents)} documents')

loaded 5 documents


Splitting document into chunks using RecursiveCharacterSplitter

In [17]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=250)
chunks=text_splitter.split_documents(documents)

print(f'Divided into {len(chunks)} chunks')
print(f'first chunk: \n\n{chunks[0]}')  

Divided into 8 chunks
first chunk: 

page_content='# About RivanCyber

RivanCyber Training Institute, Inc. is a premier IT training provider based in the Philippines. Established in the year 2000, the institute has built a reputation for excellence in delivering hands-on, mentor-led IT education.

### Mission
To provide "zero-to-hero" IT training that bridges the gap between theoretical knowledge and real-world application. We focus on job-ready skills, equipping students and professionals with the expertise needed to excel in networking, cybersecurity, and modern software development.

### Our Impact
*   **Established:** Since 2000
*   **Learners Trained:** 20,000+
*   **Certification Pass Rate:** 99% (for students who completed training and sat the exam)' metadata={'source': 'knowledge_base\\company\\about.md', 'doc_type': 'company'}


if using llamaindex

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)

chunks = text_splitter.get_nodes_from_documents(documents)

print(f'Divided into {len(chunks)} chunks')
print(f'first chunk: \n\n{chunks[0].text}')